# X01 A Mathematical Framework for Transformer Circuits


## Goal

Provides a single language for residual streams, attention heads, composition, and path expansion, and acts as the shared frame for later tracing, editing, and auditing work.


## Why this paper matters now

After M06, this moves you from 'I can read one graph' to 'I can speak the general framework.'


## Notebook and deliverable

- Source: https://transformer-circuits.pub/2021/framework/index.html
- Notebook: `notebooks/extensions/en/x01_transformer_circuits_framework.ipynb`
- Prereqs: M06
- Reproduce one minimal residual-composition toy in Colab, then write a one-page framework brief that restates one M06 toy trace using residual-stream and composition language.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

tokens = ["subject", "verb", "object"]
residual = np.array([
    [1.0, 0.2, 0.0],
    [0.3, 1.0, 0.1],
    [0.0, 0.4, 1.0],
])
attention_map = np.array([
    [0.0, 0.5, 0.0],
    [0.0, 0.0, 0.8],
    [0.7, 0.0, 0.0],
])
mlp_map = np.array([
    [0.2, 0.0, 0.1],
    [0.1, 0.3, 0.0],
    [0.0, 0.2, 0.4],
])
readout = np.array([
    [1.0, 0.2],
    [0.1, 0.9],
    [0.6, 0.4],
])

after_attention = residual + attention_map @ residual
after_mlp = after_attention + after_attention @ mlp_map
logits = after_mlp @ readout
composition_gain = logits - residual @ readout

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(residual, cmap="Blues")
axes[0].set_title("Residual stream")
axes[1].imshow(after_attention, cmap="Blues")
axes[1].set_title("After attention composition")
axes[2].imshow(after_mlp, cmap="Blues")
axes[2].set_title("After MLP composition")
for ax in axes:
    ax.set_xticks(range(3), tokens)
    ax.set_yticks(range(3), tokens)
plt.tight_layout()

print("Readout before composition:\n", np.round(residual @ readout, 3))
print("Readout after composition:\n", np.round(logits, 3))
print("Gain from composition:\n", np.round(composition_gain, 3))


## Takeaway

The point of the framework is not to list parts. It is to describe how information composes through the residual stream.


## Colab-first replication workflow


### Before you run

- Write one prediction about the mechanism before you run the cells.
- Name the baseline you are comparing against.
- Decide what result would count as a failure of your favorite story.


### After you run

- Separate observation from inference in your notes.
- Mark one ambiguity that still remains after the reproduction.
- Write one next experiment that would reduce that ambiguity.


### Ship these outputs

- Reproduce one minimal residual-composition toy in Colab, then write a one-page framework brief that restates one M06 toy trace using residual-stream and composition language.
- One experiment log with the exact settings you changed.
- One paragraph called 'what this reproduction still does not prove'.


## Self-check questions

- Why is a residual-stream view better than a layer-local description for cross-module explanations?
- In your toy reproduction, which composition step actually changed the final readout?
- If an explanation can only list head names but cannot describe composition, what is it missing?
- If you cannot answer at least two questions without reopening the notebook, rerun the reproduction and rewrite the memo.
